# ACL C++ 算子调用：GEMM

## 4.1 实验概览

按顺序运行下面的代码，需要修改的参数已在代码中标出。


In [ ]:
%%bash
set -e
echo "CANN 目录：$ASCEND_HOME_PATH"
cmake --version
npu-smi info


<a id="s-04-02_acl_runtime_and_resources"></a>


## 4.2 ACL Runtime 与资源

### 本节学习目标

- 理解 ACL 初始化和资源所有权
- 识别 RAII 清理顺序

### 先分清这些对象

| 对象 | 作用 |
|---|---|
| DeviceBuffer | 保存 NPU 上实际的矩阵数值 |
| Tensor / descriptor | 描述形状、类型和数据地址 |
| Stream | 按顺序提交和执行任务 |
| Context | 关联当前设备与运行资源 |
| workspace | 算子临时使用的内存 |
| executor | 一次算子执行所需的内部准备信息 |

示例用 C++ 对象管理资源：对象退出作用域时自动释放，这种写法称为 RAII。


### 生命周期

Runtime 负责 `aclInit → SetDevice → CreateContext → SetCurrentContext → CreateStream`。
DeviceBuffer 和 Tensor 在它创建之后创建，退出时先释放这些对象，再销毁 Stream、Context，最后 ResetDevice、Finalize。

如果 Runtime 构造到一半失败，构造函数先清理已经取得的资源再抛出异常；此时不能依赖该对象的析构函数。

### 内存与错误处理

DeviceBuffer 封装 `aclrtMalloc/aclrtFree`，copy helper 使用 `aclrtMemcpy`。宏将非成功状态转为异常，使失败不会静默继续。

### 课后实践

列出构造和析构顺序，并解释为何 stream 必须在 context/finalize 之前销毁。

参考答案见 `answer/04.02_answer.md`。

<a id="s-04-03_aclnn_gemm"></a>


## 4.3 ACLNN GEMM

### 本节学习目标

- 掌握 Tensor/Workspace/Executor 流程
- 验证 GEMM 结果

本例计算 `output = alpha × A × B + beta × C`。A 为 M×K，B 为 K×N，C 与 output 均为 M×N。本例设 alpha=1、beta=0，C 初始化为零，output 使用另一块缓冲区。


### 调用流程与计时

创建 A、B、C、output 四个 Tensor。每次执行先用 `aclnnGemmGetWorkspaceSize` 获得工作区大小和 executor，分配工作区，再执行 GEMM、同步并释放工作区。

当前时间包含上述整段操作，排除初始 H2D 和最终 D2H。它适合比较相同调用方式的参数变化；纯 kernel 时间应从 profiler 读取。

`Workspace bytes=0` 也是有效查询结果，表示本次不需要额外临时内存。每次 `execute()` 同步完成后，局部 workspace 对象就释放；所有重复执行结束后才回读 output。

本例没有开启 executor 复用，执行器由框架管理，无需额外销毁。参见[官方接口说明](https://www.hiascend.com/doc_center/source/zh/CANNCommunityEdition/910beta3/API/aolapi/operatorlist_00021.html)。


### 正确性

Host 固定随机种子生成 FP32 A/B，以三重循环计算 reference。`cubeMathType=0` 保持 FP32，最后计算相对 L2 误差。

### 编译并运行 GEMM

下面用 Device 0 计算 512×512 的矩阵乘。检查输出中的后端、设备编号和相对误差。

In [ ]:
%%bash
set -e
cd src/acl_operator_calls/GEMM-acl
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DACL_C_STUB=OFF
cmake --build build -j
# 学生修改：--m、--n、--k 分别控制输出行数、输出列数和归约维。
./build/bin/gemm_acl --m 512 --n 512 --k 512 --warmup 3 --repeat 10 --device 0


### 检查结果

输出应包含平均时间、矩阵维度 M/K/N 和相对 L2 误差。先检查误差低于 `1e-6`，再分析时间。

### 课后实践

将 GEMM 调用步骤按资源创建、算子准备、执行、同步、验证和释放分类。

参考答案见 `answer/04.03_answer.md`。